# MP2 Part 1: Commit retrieval (rkabarwa)
Retrieves all commits for my 10 assigned projects from World of Code and basic repository info from GitHub.

In [ ]:
# !python -m pip install -U python-woc pandas matplotlib tqdm requests tabulate
import json, os, time
import pandas as pd
import requests
from tqdm import tqdm
from woc.remote import WocMapsRemote

woc = WocMapsRemote(base_url="https://worldofcode.org/api/")
netid = 'rkabarwa'

projects = pd.read_csv('net2prj.csv')
projects = projects[projects.netID == netid].reset_index(drop=True)
projects

## Step 1: commit sha1s for each project (p2c)

In [ ]:
dfs = []
for prj in projects.WoC:
    shas = woc.get_values('p2c', prj)
    print(prj, len(shas))
    dfs.append(pd.DataFrame({'sha1': shas, 'project': prj}))
    time.sleep(1)
df = pd.concat(dfs, ignore_index=True)
df.to_csv('df_commits.csv', index=False)
print('total', len(df))

## Step 2: commit content in batches of 40 (under the 50 limit), 1s wait between requests
Results are cached in `commit_cache.jsonl` so the cell can be re-run after an interruption without re-downloading.

In [ ]:
CACHE = 'commit_cache.jsonl'
done = {}
if os.path.exists(CACHE):
    with open(CACHE) as f:
        for line in f:
            r = json.loads(line)
            done[r['commit']] = r

todo = [s for s in df.sha1.unique() if s not in done]
chunks = [todo[i:i+40] for i in range(0, len(todo), 40)]
missing = {}
with open(CACHE, 'a') as f:
    for chunk in tqdm(chunks):
        for attempt in range(3):
            try:
                res, err = woc.get_values_many('commit.tch', chunk)
                break
            except Exception as e:
                print('retrying after error:', e)
                time.sleep(10)
        else:
            raise RuntimeError('batch failed 3 times')
        missing.update(err or {})
        for sha, v in res.items():
            c = v[0]  # [tree, parents, [author, time, tz], [committer, time, tz], message]
            r = {'commit': sha, 'author': c[2][0], 'time': int(c[2][1]), 'message': c[4]}
            done[sha] = r
            f.write(json.dumps(r) + '\n')
        time.sleep(1)

print('retrieved', len(done), 'of', df.sha1.nunique(), 'unique commits;',
      len(missing), 'not found in WoC commit.tch in this run')

## Step 3: write `rkabarwa_project_summary.csv` (semicolon separated)

In [ ]:
dfc = df.merge(pd.DataFrame(done.values()), left_on='sha1', right_on='commit')
summary = dfc[['project', 'sha1', 'author', 'time', 'message']]
summary.columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
summary.to_csv(netid + '_project_summary.csv', sep=';', index=False)
print(len(summary), 'rows written')
summary.head()

## Step 4: WoC stats per project

In [ ]:
stats = summary.groupby('project_wocid').agg(
    ncommits=('commit_sha1', 'count'),
    nauthors=('author', 'nunique'),
    min_time=('time', 'min'),
    max_time=('time', 'max'))
stats['min_time'] = pd.to_datetime(stats.min_time, unit='s')
stats['max_time'] = pd.to_datetime(stats.max_time, unit='s')
stats = stats.reindex(projects.WoC)
print(stats.to_markdown())

## Step 5: GitHub stats (stars, forks, last commit date)

In [ ]:
gh = []
for wid, url in zip(projects.WoC, projects.GH):
    repo = url.replace('https://github.com/', '')
    r = requests.get(f'https://api.github.com/repos/{repo}').json()
    c = requests.get(f'https://api.github.com/repos/{repo}/commits?per_page=1').json()
    gh.append({'project': wid, 'repo': repo,
               'nstars': r['stargazers_count'], 'nforks': r['forks_count'],
               'lastGHCommitDate': c[0]['commit']['committer']['date'][:10]})
    time.sleep(1)
gh = pd.DataFrame(gh)
print(gh.to_markdown(index=False))